# X01 Transformer Circuits 的数学框架


## 目标

给 residual stream、attention head、composition 和 path expansion 一个统一语言，是后续 tracing、editing 和 auditing 的总框架。


## 为什么现在学这篇

当你已经做完 M06 之后，这篇能把你从“会读一张图”推进到“会说一个一般框架”。


## Notebook 与交付

- 原文: https://transformer-circuits.pub/2021/framework/index.html
- Notebook：`notebooks/extensions/zh/x01_transformer_circuits_framework.ipynb`
- 先修: M06
- 在 Colab 里复现一个最小 residual-composition toy，并写 1 页 framework brief，把 M06 里的一个 toy trace 用 residual-stream 与 composition 语言重述。


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Jonny-English/learn-interpretability.git"
REPO_DIR = "learn-interpretability"
REPO_BRANCH = "main"

if "google.colab" in sys.modules:
    candidate = Path("/content") / REPO_DIR
    if not candidate.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(candidate)],
            check=True,
        )
    os.chdir(candidate)

root = Path.cwd().resolve()
while not (root / "content" / "course.json").exists():
    if root.parent == root:
        raise RuntimeError("Run this notebook from the repository root.")
    root = root.parent

import matplotlib.pyplot as plt
import numpy as np

tokens = ["subject", "verb", "object"]
residual = np.array([
    [1.0, 0.2, 0.0],
    [0.3, 1.0, 0.1],
    [0.0, 0.4, 1.0],
])
attention_map = np.array([
    [0.0, 0.5, 0.0],
    [0.0, 0.0, 0.8],
    [0.7, 0.0, 0.0],
])
mlp_map = np.array([
    [0.2, 0.0, 0.1],
    [0.1, 0.3, 0.0],
    [0.0, 0.2, 0.4],
])
readout = np.array([
    [1.0, 0.2],
    [0.1, 0.9],
    [0.6, 0.4],
])

after_attention = residual + attention_map @ residual
after_mlp = after_attention + after_attention @ mlp_map
logits = after_mlp @ readout
composition_gain = logits - residual @ readout

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
axes[0].imshow(residual, cmap="Blues")
axes[0].set_title("Residual stream")
axes[1].imshow(after_attention, cmap="Blues")
axes[1].set_title("After attention composition")
axes[2].imshow(after_mlp, cmap="Blues")
axes[2].set_title("After MLP composition")
for ax in axes:
    ax.set_xticks(range(3), tokens)
    ax.set_yticks(range(3), tokens)
plt.tight_layout()

print("Readout before composition:\n", np.round(residual @ readout, 3))
print("Readout after composition:\n", np.round(logits, 3))
print("Gain from composition:\n", np.round(composition_gain, 3))


## 小结

这篇的关键不是给部件起名字，而是学会用 residual stream 语言描述信息如何组合起来。


## Colab 优先的复现流程


### 运行前

- 在运行前先写 1 条你对机制的预测。
- 先写清你要对比的 baseline 是什么。
- 先决定什么结果会推翻你最喜欢的解释。


### 运行后

- 在笔记里把 observation 和 inference 分开。
- 标出复现之后仍然存在的 1 个歧义。
- 写 1 个能降低该歧义的下一步实验。


### 最后交付这些产物

- 在 Colab 里复现一个最小 residual-composition toy，并写 1 页 framework brief，把 M06 里的一个 toy trace 用 residual-stream 与 composition 语言重述。
- 1 份 experiment log，写清你改了哪些设置。
- 1 段“这次复现仍然不能证明什么”。


## 验收题

- 为什么 residual stream 视角比逐层局部描述更适合做跨模块解释？
- 在你的 toy 复现里，哪一次 composition 真的改变了最终读出？
- 如果一个解释只会列 head 名称、不会说 composition，它缺了什么？
- 如果你不能在不重开 notebook 的情况下独立答出至少 2 题，就回去重跑复现并重写 memo。
